In [19]:
# Cryptarithmetic Puzzle using CSP

letters = ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y']

# Carry variables
carries = ['C1', 'C2', 'C3', 'C4']

variables = letters + carries

domains = {letter: set(range(10)) for letter in letters}

# Leading zero is not allowed
domains['S'].remove(0)
domains['M'].remove(0)

# Carry domains
for c in carries:
    domains[c] = {0, 1}

# MONEY is a five-digit number.
# Therefore C4 = 1 and C4 = M.
domains['C4'] = {1}
domains['M'] = {1}

In [20]:
def check_constraints(assignment):

    # All letters must have unique digits
    if len(set(assignment.values())) != len(assignment):
        return False

    # Leading zero constraint
    if assignment.get('S', 1) == 0:
        return False

    if assignment.get('M', 1) == 0:
        return False

    # Check arithmetic when all letters are assigned
    if len(assignment) == len(letters):

        SEND = (assignment['S'] * 1000 +
                assignment['E'] * 100 +
                assignment['N'] * 10 +
                assignment['D'])

        MORE = (assignment['M'] * 1000 +
                assignment['O'] * 100 +
                assignment['R'] * 10 +
                assignment['E'])

        MONEY = (assignment['M'] * 10000 +
                 assignment['O'] * 1000 +
                 assignment['N'] * 100 +
                 assignment['E'] * 10 +
                 assignment['Y'])

        return SEND + MORE == MONEY

    return True

In [21]:
# Minimum Remaining Values (MRV)
def select_variable(assignment, current_domains):

    unassigned = [v for v in letters if v not in assignment]

    return min(unassigned, key=lambda v: len(current_domains[v]))


# Least Constraining Value (LCV)
def order_values(variable, assignment, current_domains):

    values = list(current_domains[variable])

    def conflict_count(value):

        conflicts = 0

        for other in letters:

            if other != variable and other not in assignment:

                if value in current_domains[other]:
                    conflicts += 1

        return conflicts

    return sorted(values, key=conflict_count)

In [22]:
# Forward Checking
def forward_check(variable, value, assignment, current_domains):

    new_domains = {
        v: set(current_domains[v])
        for v in variables
    }

    new_domains[variable] = {value}

    # Remove assigned digit from other letters
    for other in letters:

        if other not in assignment and other != variable:

            new_domains[other].discard(value)

    # Check if any unassigned letter has an empty domain
    for other in letters:

        if other not in assignment and other != variable:

            if not new_domains[other]:
                return None

    return new_domains

In [23]:
# Arithmetic constraints
constraints = [

    (['D', 'E', 'Y', 'C1'],
     lambda d, e, y, c1: d + e == y + 10*c1),

    (['N', 'R', 'C1', 'E', 'C2'],
     lambda n, r, c1, e, c2: n + r + c1 == e + 10*c2),

    (['E', 'O', 'C2', 'N', 'C3'],
     lambda e, o, c2, n, c3: e + o + c2 == n + 10*c3),

    (['S', 'M', 'C3', 'O', 'C4'],
     lambda s, m, c3, o, c4: s + m + c3 == o + 10*c4)
]


# Direct deductions when enough values are known
def direct_deductions(d):

    # Units column: D + E = Y + 10*C1
    if len(d['D']) == 1 and len(d['E']) == 1:

        total = next(iter(d['D'])) + next(iter(d['E']))

        y = total % 10
        c1 = total // 10

        if y not in d['Y'] or c1 not in d['C1']:
            return False

        d['Y'] = {y}
        d['C1'] = {c1}


    # Tens column: N + R + C1 = E + 10*C2
    if all(len(d[v]) == 1 for v in ['N', 'R', 'C1', 'E']):

        n, r, c1, e = [next(iter(d[v]))
                       for v in ['N', 'R', 'C1', 'E']]

        total = n + r + c1

        c2 = total // 10

        if total % 10 != e or c2 not in d['C2']:
            return False

        d['C2'] = {c2}


    # Hundreds column: E + O + C2 = N + 10*C3
    if all(len(d[v]) == 1 for v in ['E', 'O', 'C2', 'N']):

        e, o, c2, n = [next(iter(d[v]))
                       for v in ['E', 'O', 'C2', 'N']]

        total = e + o + c2

        c3 = total // 10

        if total % 10 != n or c3 not in d['C3']:
            return False

        d['C3'] = {c3}


    # Thousands column: S + M + C3 = O + 10*C4
    if all(len(d[v]) == 1 for v in ['S', 'M', 'C3', 'C4']):

        s, m, c3, c4 = [next(iter(d[v]))
                        for v in ['S', 'M', 'C3', 'C4']]

        total = s + m + c3

        o = total - 10*c4

        if o not in d['O']:
            return False

        d['O'] = {o}


    # Reverse deduction: calculate M if S, O, C3 and C4 are known
    if all(len(d[v]) == 1 for v in ['S', 'O', 'C3', 'C4']):

        s, o, c3, c4 = [next(iter(d[v]))
                        for v in ['S', 'O', 'C3', 'C4']]

        m = o + 10*c4 - s - c3

        if m not in d['M']:
            return False

        d['M'] = {m}

    return True


# GAC: Remove values that have no supporting combination
def revise(scope, condition, d):

    supported = {v: set() for v in scope}

    for values in product(*(d[v] for v in scope)):

        if condition(*values):

            for variable, value in zip(scope, values):
                supported[variable].add(value)

    for variable in scope:

        d[variable] &= supported[variable]

        if not d[variable]:
            return False

    return True


from itertools import product


# Constraint Propagation
def propagate(assignment, current_domains):

    changed = True

    while changed:

        before = {
            v: set(current_domains[v])
            for v in variables
        }

        # Keep assigned variables fixed
        for variable, value in assignment.items():
            current_domains[variable] = {value}

        # Direct arithmetic deductions
        if not direct_deductions(current_domains):
            return False

        # GAC on arithmetic constraints
        for scope, condition in constraints:

            if not revise(scope, condition, current_domains):
                return False

        # AllDifferent constraint
        assigned_values = set(assignment.values())

        for variable in letters:

            if variable not in assignment:

                current_domains[variable] -= assigned_values

                if not current_domains[variable]:
                    return False

        # Repeat if any domain changed
        changed = any(
            before[v] != current_domains[v]
            for v in variables
        )

    return True

In [24]:
# Backtracking Search
def backtracking(assignment, current_domains):

    if len(assignment) == len(letters):

        if check_constraints(assignment):
            return assignment

        return None

    variable = select_variable(assignment, current_domains)

    for value in order_values(variable, assignment, current_domains):

        if value in assignment.values():
            continue

        assignment[variable] = value

        if check_constraints(assignment):

            new_domains = forward_check(
                variable, value, assignment, current_domains
            )

            if new_domains is not None:

                if propagate(assignment, new_domains):

                    result = backtracking(assignment, new_domains)

                    if result is not None:
                        return result

        # Backtrack
        del assignment[variable]

    return None

In [37]:
# Execute the CSP solver

initial_assignment = {}

solution = backtracking(
    initial_assignment,
    domains
)

# Display the result
if solution:

    print("Solution found:")

    for letter in letters:
        print(letter, "=", solution[letter])

    SEND = (solution['S'] * 1000 +
            solution['E'] * 100 +
            solution['N'] * 10 +
            solution['D'])

    MORE = (solution['M'] * 1000 +
            solution['O'] * 100 +
            solution['R'] * 10 +
            solution['E'])

    MONEY = (solution['M'] * 10000 +
             solution['O'] * 1000 +
             solution['N'] * 100 +
             solution['E'] * 10 +
             solution['Y'])

    print("\nSEND =", SEND)
    print("MORE =", MORE)
    print("MONEY =", MONEY)

    print("\nVerification:")
    print(SEND, "+", MORE, "=", MONEY)

else:
    print("No solution found.")

Solution found:
S = 9
E = 5
N = 6
D = 7
M = 1
O = 0
R = 8
Y = 2

SEND = 9567
MORE = 1085
MONEY = 10652

Verification:
9567 + 1085 = 10652
